In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
class Sketcher:
    def __init__(self, windowname, dests, colors_func):
        self.prev_pt = None
        self.dests = dests
        self.colors_func = colors_func
        self.windowname = windowname
        self.show()
        cv2.setMouseCallback(self.windowname, self.on_mouse)

    def show(self):
        cv2.imshow(self.windowname, self.dests[0])

    def on_mouse(self, event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            self.prev_pt = (x, y)
            # radius is 15
            # Use sobel operators to find patch with similar gradient
            # do seamless cloning on the original patch of the image
            pass
        elif event == cv2.EVENT_LBUTTONUP:
            self.prev_pt = None

In [2]:
import cv2
import numpy as np

def selectBlemish(x, y, r):
    global i
    cropImg = src[y: (y+2*r), x: (x+2*r)]

    return identifyBestPatch(x,y,r)

def identifyBestPatch(x,y,r):
    patches = {}

    key1 = appendDictionary(x+2*r, y)
    patches['Key 1'] = (x+2*r, y, key1[0], key1[1])

    key2 = appendDictionary(x+2*r, y+r)
    patches['Key 2'] = (x+2*r, y+r, key2[0], key2[1])

    key3 = appendDictionary(x-2*r, y)
    patches['Key 3'] = (x-2*r, y, key3[0], key3[1])

    key4 = appendDictionary(x-2*r, y-r)
    patches['Key 4'] = (x-2*r, y-r, key4[0], key4[1])

    key5 = appendDictionary(x, y+2*r)
    patches['Key 5'] = (x, y+2*r, key5[0], key5[1])

    key6 = appendDictionary(x+r, y+2*r)
    patches['Key 6'] = (x+r, y+2*r, key6[0], key6[1])

    key7 = appendDictionary(x, y-2*r)
    patches['Key 7'] = (x, y-2*r, key7[0], key7[1])

    key8 = appendDictionary(x-r, y-2*r)
    patches['Key 8'] = (x-r, y-2*r, key8[0], key8[1])

    # print(patches)
    findLowX = {}
    findLowY = {}

    for key, (x, y, gx, gy) in patches.items():
        findLowX[key] = gx
        findLowY[key] = gy

    keyMinX = min(findLowX.keys(), key = (lambda k: findLowX[k]))
    keyMinY = min(findLowY.keys(), key = (lambda k: findLowY[k]))

    if keyMinX == keyMinY:
        return patches[keyMinX][0], patches[keyMinX][1]
    else:
        return patches[keyMinX][0], patches[keyMinX][1] # try keyMinY
    
def appendDictionary(x, y):
    cropImg = src[y: (y+2*r), x: (x+2*r)]
    gx, gy = sobelFilter(cropImg)
    return gx, gy

def sobelFilter(cropImg):
    sobelx = cv2.Sobel(cropImg, cv2.CV_64F, 1, 0, ksize = 5)
    absSobelx = np.absolute(sobelx)
    gx = np.mean(np.uint8(absSobelx))

    sobely = cv2.Sobel(cropImg, cv2.CV_64F, 0, 1, ksize = 5)
    absSobely = np.absolute(sobely)
    gy = np.mean(np.uint8(absSobely))

    return gx, gy

def blemishRemoval(action, x, y, flags, param):
    global r, src

    if action == cv2.EVENT_LBUTTONDOWN:
        blemishLocation = (x, y)

        newX, newY = selectBlemish(x, y, r)
        newPatch = src[newY: (newY+2*r), newX: (newX+2*r)]
        cv2.imwrite("data/newPatch.png", newPatch)
        mask = 255 * np.ones(newPatch.shape, newPatch.dtype)
        src = cv2.seamlessClone(newPatch, src, mask, blemishLocation, cv2.NORMAL_CLONE)
        cv2.imshow("Blemish Removal Tool", src)

    elif action == cv2.EVENT_LBUTTONUP:
        cv2.imshow("Blemish Removal Tool", src)


r = 15
i = 0
src = cv2.imread("data/blemish.png")
dummy =  src.copy()
cv2.namedWindow("Blemish Removal Tool")
cv2.setMouseCallback("Blemish Removal Tool", blemishRemoval)
k=0
while k != 27:
    cv2.imshow("Blemish Removal Tool", src)
    k = cv2.waitKey(20) & 0xFF
    if k == 99:
        src = dummy.copy()

cv2.destroyAllWindows()